In [1]:
import os
from langchain_deepseek import ChatDeepSeek
chat = ChatDeepSeek(
    model="deepseek-chat",   # DeepSeek-V3: 支持 tools/structured output
    temperature=0,
    api_key="sk-3d4dbf63d7704840976b05ddbb9957a7"
)


/hard_data1/user/yangguobin/anaconda3/envs/retriever/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field, field_validator

In [9]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field, field_validator

class Joke(BaseModel):
    setup: str = Field(description="笑话铺垫（请以问号结尾）")
    punchline: str = Field(description="笑话包袱")

    @field_validator("setup")
    @classmethod
    def setup_must_end_with_qmark(cls, v: str) -> str:
        if not v.endswith("？"):
            print("格式存在错误")
        return v

#创建解析器
parser = PydanticOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
    template=(
        "请根据用户输入生成一个笑话，并严格按要求输出。\n"
        "{format_instructions}\n"
        "用户输入：{query}\n"
    ),
    input_variables=["query"],
    #get_format_instructions会生成一段提示词
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | chat | parser

# ✅ invoke 必须传 dict
result = chain.invoke({"query": "讲一个笑话"})
print(result)

setup='为什么程序员总是分不清万圣节和圣诞节？' punchline='因为 Oct 31 等于 Dec 25。'


In [10]:

from langchain_core.output_parsers import JsonOutputParser
parser=JsonOutputParser(pydantic_object=Joke)
chain=prompt|chat|parser
chain.invoke({"query":"讲一个笑话"})

{'setup': '为什么程序员总是分不清万圣节和圣诞节？', 'punchline': '因为 Oct 31 等于 Dec 25。'}

In [12]:
from typing import Iterable
from langchain_core.messages import AIMessage,AIMessageChunk
def parse(ai_message:AIMessage)->str:
    return ai_message.content.swapcase()

chain=chat|parse
chain.invoke("hello,请用英文回答")

"hELLO! i'M HERE AND READY TO HELP. pLEASE FEEL FREE TO ASK YOUR QUESTION IN eNGLISH, AND i'LL DO MY BEST TO ASSIST YOU."